## Frontier Model APIs

In this notebook, I explored not only the frontier paid API models but also Google Gemini and the router model `z-ai/glm-4.6v`. I also downloaded the Ollama `gpt-oss:20b` model, but it took noticeably longer to run on some of the tasks.

Aside from the cost and latency of each model, what caught my attention was the `reasoning_effort` argument and its effectiveness in improving response accuracy when switching it from `minimal` to `low`.

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

### Load API Keys

Load environment variables from `.env` and check which API keys are available. `OPENAI_API_KEY` is required; the rest (`GOOGLE_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`) are optional and only needed if you're testing those providers.

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")`

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-


### Set Up Clients

Create OpenAI client instances for each provider. Since OpenAI's client is just a thin wrapper around HTTP calls, and Gemini, Groq, and OpenRouter all expose OpenAI-compatible endpoints, we can reuse the same client by simply pointing `base_url` at each provider (same idea for local models via Ollama).

In [ ]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

##### Checking whether Ollama is running locally


In [15]:
requests.get("http://localhost:11434/").content

b'Ollama is running'

##### Downloading llama3.2

In [16]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕███████████████

##### Downloading `gpt-oss:20b`

Pull the model locally via Ollama (this may take a while depending on your connection).

In [17]:
!ollama pull gpt-oss:20b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 681 KB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 1.3 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 2.4 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 3.8 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 5.0 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 6.0 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕           

### Comparing Providers: Telling a Joke

Send the same prompt to OpenAI, Gemini, and the local Ollama model to compare their responses.

In [4]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [5]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM engineer bring a ladder to the training session?

Because they heard the model needed more layers!

In [8]:
response = gemini.chat.completions.create(model="gemini-flash-latest", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Here is a joke for your journey:

***

A mother asks her son, who is studying to become an LLM Engineer, how his final exams went.

"I did amazing!" the student replies. "I answered every single question with absolute authority, flawless grammar, and professional formatting."

"That's wonderful!" his mother says. "Did you get them all right?"

The student shrugs. "Oh, I have absolutely no idea. I was hallucinating the entire time, but my confidence score was 99.8%."

***

**Bonus advice for your journey:** 
When studying gets tough, just remember: *You aren't failing, you are just in the early epochs of your training run. Keep your learning rate high and try not to overfit!*

In [20]:
response = ollama.chat.completions.create(model="gpt-oss:20b", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

What does an aspiring LLM engineer do the night before starting a big training run?

—Practices **multi‑head attention** on their notes (makes sure every key point gets its own focus!).

### Training vs Inference time scaling

In [9]:
puzzle = [
    {"role": "user", "content": 
        "You toss 3 coins. two of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [10]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/3

In [11]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

3/4

In [14]:
response = gemini.chat.completions.create(model="gemini-flash-latest", messages=puzzle, reasoning_effort= "minimal")
display(Markdown(response.choices[0].message.content))

3/4

#### Gemini Client Library

In [25]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-flash-latest", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Blue is the feeling of dipping your hands into cool, calm water—a soothing, spacious sensation of quiet peace.


### Routers and Abtraction Layers

In [28]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.6v", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Of course! Here's a joke for a student on the journey to becoming an expert in LLM Engineering:

**Joke:**

A student LLM engineer and an expert LLM engineer are both trying to get their model to solve a complex problem.

The student spends weeks meticulously writing code, optimizing the training loop, and tuning hyperparameters.

The expert spends 10 minutes writing a perfect prompt.

The student asks, "How did you do that so fast?"

The expert replies, "I didn't have to. I just asked the model to write the code for me."

***

**Why it's funny (and relevant to your journey):**

This joke highlights a key insight in the field: as you gain experience, your focus shifts. Early on, the biggest challenge is often the code and the training process itself. As you become an expert, you realize that the *input*—the prompt—is where the real magic (and the real work) happens. You learn to speak the language of the model effectively, which can save you countless hours of debugging.

Keep learning, and soon you'll be the one laughing at your own old, overly-complicated code!